# Version Ollama locale

Cette copie du projet utilise Gradio + Ollama local au lieu de Streamlit/OpenAI API. Lancez `python gradio_app.py` après avoir installé les modèles avec `ollama pull qwen2.5:3b`, `ollama pull llama3.2:3b`, `ollama pull mistral:7b`.


# Week 3 - Pipeline RAG

Ce notebook implemente le pipeline RAG demande pour le corpus `export_final.csv` : recherche des documents pertinents, injection dans le prompt, generation contextualisee, comparaison de 3 LLMs, evaluation precision/recall, detection hors domaine et interface web Streamlit.

## 1. Chargement du corpus

Le corpus contient les articles extraits du code de la route. Chaque ligne represente un document recuperable par le systeme RAG.

In [22]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('export_final.csv')
df = pd.read_csv(DATA_PATH)

print(f'Nombre d\'articles: {len(df)}')
df.head()

Nombre d'articles: 266


,article_id,infraction_desc,categorie_vehicule,amende_fixe,amende_min,amende_max,points_retrait,mots_cles,role_paragraphe,has_prison,has_license_penalty,source,model_approach,cluster_theme
0,1,ال يجوز الي شخص ان يسوق مركبه ذات محرك او مجمو...,vehicule leger / voiture,NaN,NaN,NaN,NaN,permis,autre,False,False,code de la route MA52_05.pdf,pretrained_sentence_transformer,5
1,2,استثناء من احكام الماده االولي اعاله : - يجوز ...,non specifie,NaN,NaN,NaN,NaN,permis,obligation,False,False,code de la route MA52_05.pdf,pretrained_sentence_transformer,0
2,4,في حاله السير الدولي ووفقا لالتفاقيه الدوليه ل...,non specifie,NaN,NaN,NaN,NaN,"permis, depassement",autre,False,False,code de la route MA52_05.pdf,pretrained_sentence_transformer,5
3,5,من 13 بتاريخ 1.16.106 الصادر بتنفيذه الظهري ال...,vehicule leger / voiture,NaN,NaN,NaN,NaN,"permis, alcool",autre,False,False,code de la route MA52_05.pdf,pretrained_sentence_transformer,4
4,7,من 13 بتاريخ 1.16.106 الصادر بتنفيذه الظهري ال...,moto / deux roues; poids lourd / marchandises;...,NaN,NaN,NaN,NaN,"permis, depassement, chargement",autre,False,False,code de la route MA52_05.pdf,pretrained_sentence_transformer,2


In [ ]:
df[['article_id', 'categorie_vehicule', 'mots_cles', 'role_paragraphe', 'source', 'infraction_desc']]

## 2. Initialisation du pipeline RAG

Le coeur du pipeline est dans `rag_core.py`. Il reste local et reproductible : il utilise TF-IDF pour la recherche documentaire, ce qui evite les erreurs de telechargement de modeles pendant la demonstration.

In [23]:
from rag_core import RAGPipeline, parse_expected_ids

pipeline = RAGPipeline(DATA_PATH)
print('Pipeline charge.')
print('Taille du corpus:', len(pipeline.df))

Pipeline charge.
Taille du corpus: 266


## 3. Reception d'une question utilisateur

On definit une question. Vous pouvez modifier cette variable pour tester le systeme.

In [24]:
question = 'Quelles sont les regles concernant le permis de conduire ?'
question

'Quelles sont les regles concernant le permis de conduire ?'

## 4. Recherche des documents les plus pertinents

La fonction `retrieve` retourne les articles les plus proches de la question, avec un score de similarite.

In [25]:
documents = pipeline.retrieve(question, k=3)

retrieval_table = pd.DataFrame([
    {
        'article_id': doc.article_id,
        'score': round(doc.score, 4),
        'mots_cles': doc.keywords,
        'source': doc.source,
        'extrait': doc.text[:250] + ('...' if len(doc.text) > 250 else ''),
    }
    for doc in documents
])
retrieval_table

,article_id,score,mots_cles,source,extrait
0,250,0.0756,permis,code de la route MA52_05.pdf,الطرقيه، يجب علي الحاصلين علي رخصه باستغالل مو...
1,33,0.0750,"permis, depassement",code de la route MA52_05.pdf,) نقط وذلك 4( يجوز لصاحب رخصه السياقه، قبل انص...
2,178,0.0716,"permis, depassement, chargement",code de la route MA52_05.pdf,دون االخالل بالعقوبات االشد، يعاقب عن تجاوز ال...


## 5. Injection des documents dans le prompt

Les documents retrouves sont injectes dans un prompt. Le generateur doit repondre uniquement avec ce contexte.

In [26]:
prompt = pipeline.build_prompt(question, documents)
print(prompt)

Tu es un assistant juridique RAG. Reponds uniquement avec le contexte fourni. Si le contexte est insuffisant, dis-le clairement.

Question utilisateur:
Quelles sont les regles concernant le permis de conduire ?

Documents pertinents:
[Article 250 | score=0.076 | source=code de la route MA52_05.pdf]
الطرقيه، يجب علي الحاصلين علي رخصه باستغالل موسسه لتعليم السياقه او للتربيه علي السالمه قبل توقيف او انهاء نشاطهم، اخبار االداره بذلك، بواسطه رساله مضمونه مع االشعار باالستالم.

[Article 33 | score=0.075 | source=code de la route MA52_05.pdf]
) نقط وذلك 4( يجوز لصاحب رخصه السياقه، قبل انصرام الفتره االختباريه، ان يسترجع اربع ضي المخصص لرخصته، اذا خضع لدوره في التربيه علي السالمه الطرقيه.�دون تجاوز الحد االق

[Article 178 | score=0.072 | source=code de la route MA52_05.pdf]
دون االخالل بالعقوبات االشد، يعاقب عن تجاوز الوزن الماذون به عند المرور باحدي منشات )4.000( ) الي اربعه االف 1.500( العبور، بالحبس من شهر الي ثالثه اشهر وبغرامه من الف وخمسمايه درهم عن كل طن زايد او باحدي هاتين العقوبتين ف

## 6. Generation d'une reponse contextualisee

La reponse cite les articles utilises comme references.

In [27]:
answer = pipeline.generate_answer(question, documents)
print(answer)

Le corpus associe votre question a Article 250. Contenu pertinent: الطرقيه، يجب علي الحاصلين علي رخصه باستغالل موسسه لتعليم السياقه او للتربيه علي السالمه قبل توقيف او انهاء نشاطهم، اخبار االداره بذلك، بواسطه رساله مضمونه مع االشعار باالستالم. Documents consultes: Article 250, Article 33, Article 178.


## 7. Comparaison de 3 LLMs

Le projet supporte deux modes de comparaison :

- mode rapide, sans API, avec trois generateurs locaux simules ;
- mode reel avec Qwen, GPT et Llama.

Pour le mode reel :

- Qwen utilise `transformers` avec `QWEN_MODEL`, par defaut `Qwen/Qwen2.5-1.5B-Instruct` ;
- GPT utilise l'API OpenAI avec `OPENAI_API_KEY` et `OPENAI_MODEL`, par defaut `gpt-4o-mini` ;
- Llama utilise Ollama avec `LLAMA_MODEL`, par defaut `llama3.2`.

Le meme prompt RAG est envoye aux trois modeles.

In [30]:
# Mode rapide: generateurs locaux simules
comparison = pipeline.compare_llms(question, k=3, use_real_models=False)
comparison_df = pd.DataFrame(comparison)
comparison_df

,modele,reponse,nb_references,meilleur_score
0,LLM-A extractif,Le corpus associe votre question a Article 250...,3,0.0756
1,LLM-B structure,Reponse:\nLa question est rapprochee principal...,3,0.0756
2,LLM-C prudent,"D'apres les articles retrouves, la reponse la ...",3,0.0756


### Comparaison reelle Qwen/GPT/Llama

Executez cette cellule seulement si les dependances et services sont prets. Sans `OPENAI_API_KEY` ou sans Ollama, le notebook affichera un message d'indisponibilite au lieu de planter.

In [37]:
%pip install -U openai

Defaulting to user installation because normal site-packages is not writeable
  Using cached openai-2.32.0-py3-none-any.whl.metadata (31 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.14.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.13.3-py3-none-any.whl.metadata (108 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.3-cp313-cp313-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached openai-2.32.0-py3-none-any.whl (1.2 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.14.0-cp313-cp313-win_amd64.whl (201 kB)
Using cached pydantic-2.13.3-py3-none-any.whl (471 kB)
Using cached pydantic_core-2.46.3-cp313-cp313-win_amd64.whl (2.1 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)

   ----- ------------

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\amami\\AppData\\Roaming\\Python\\Python313\\site-packages\\openai\\cli\\_cli.py'
Check the permissions.



Defaulting to user installation because normal site-packages is not writeable
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.2 MB 1.4 MB/s eta 0:00:01
   --------------------------- ------------ 0.8/1.2 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.1 MB 2.0 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 2.1 MB/s eta 0:00:01
   ------------------------------ --------- 1.6/2.1 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 2.1/

In [35]:
# Attention: cette cellule peut telecharger/charger Qwen et appeler des services externes.
# Decommentez pour tester les vrais modeles.

real_comparison = pipeline.compare_llms(question, k=3, use_real_models=True)
pd.DataFrame(real_comparison)

TypeError: RAGPipeline.compare_llms() got an unexpected keyword argument 'use_real_models'

## 8. Evaluation des performances : precision et recall

Deux modes sont possibles :

- si vous donnez des articles attendus, l'evaluation utilise cette verite terrain ;
- sinon, elle estime les articles pertinents par recouvrement lexical avec la question.

In [31]:
# Optionnel: renseigner les articles attendus, par exemple '1, 2'
expected_raw = ''
expected_ids = parse_expected_ids(expected_raw)

evaluation = pipeline.evaluate(question, documents, expected_ids)
evaluation

{'mode': 'estimation par recouvrement lexical',
 'precision': 1.0,
 'recall': 0.022,
 'retrieved_ids': ['178', '250', '33'],
 'relevant_ids': ['1',
  '10',
  '100',
  '101',
  '102',
  '103',
  '106',
  '107',
  '109',
  '11',
  '110',
  '111',
  '112',
  '113',
  '114',
  '115',
  '118',
  '12',
  '120',
  '125',
  '127',
  '128',
  '13',
  '130',
  '131',
  '133',
  '14',
  '147',
  '148',
  '149',
  '15',
  '150',
  '152',
  '153',
  '155',
  '16',
  '161',
  '166',
  '167',
  '168',
  '169',
  '170',
  '172',
  '174',
  '175',
  '178',
  '18',
  '181',
  '183',
  '184',
  '185',
  '191',
  '194',
  '198',
  '2',
  '20',
  '21',
  '215',
  '216',
  '218',
  '22',
  '224',
  '225',
  '228',
  '229',
  '23',
  '231',
  '233',
  '234',
  '238',
  '239',
  '24',
  '240',
  '243',
  '245',
  '248',
  '249',
  '250',
  '251',
  '252',
  '255',
  '256',
  '257',
  '26',
  '260',
  '262',
  '263',
  '264',
  '265',
  '267',
  '27',
  '271',
  '272',
  '274',
  '275',
  '276',
  '278',
  '27

## 9. Detection de questions hors domaine

Le systeme compare le score de similarite et les termes de domaine. Une question trop eloignee du corpus est rejetee.

In [32]:
ood_question = 'Quel est le meilleur restaurant a Casablanca ?'
ood_documents = pipeline.retrieve(ood_question, k=3)
is_ood, reason = pipeline.is_out_of_domain(ood_question, ood_documents)

print('Question:', ood_question)
print('Hors domaine:', is_ood)
print('Raison:', reason)
print(pipeline.generate_answer(ood_question, ood_documents))

Question: Quel est le meilleur restaurant a Casablanca ?
Hors domaine: True
Raison: La question semble hors domaine: elle ne correspond pas assez aux articles du code de la route disponibles.
La question semble hors domaine: elle ne correspond pas assez aux articles du code de la route disponibles. Je ne peux pas donner une reponse fiable avec ce corpus. Essayez une question sur le permis, les infractions, les vehicules ou les sanctions routieres.


## 10. Execution complete du pipeline

La methode `answer` regroupe toutes les etapes : documents, prompt, reponse, comparaison, evaluation et detection hors domaine.

In [33]:
result = pipeline.answer(
    question,
    k=3,
    expected_ids=expected_ids,
    use_real_models=False,
)

print('Question:', result['question'])
print('Hors domaine:', result['out_of_domain'])
print('Reponse:\n', result['answer'])
print('Evaluation:', result['evaluation'])

Question: Quelles sont les regles concernant le permis de conduire ?
Hors domaine: False
Reponse:
 Le corpus associe votre question a Article 250. Contenu pertinent: الطرقيه، يجب علي الحاصلين علي رخصه باستغالل موسسه لتعليم السياقه او للتربيه علي السالمه قبل توقيف او انهاء نشاطهم، اخبار االداره بذلك، بواسطه رساله مضمونه مع االشعار باالستالم. Documents consultes: Article 250, Article 33, Article 178.
Evaluation: {'mode': 'estimation par recouvrement lexical', 'precision': 1.0, 'recall': 0.022, 'retrieved_ids': ['178', '250', '33'], 'relevant_ids': ['1', '10', '100', '101', '102', '103', '106', '107', '109', '11', '110', '111', '112', '113', '114', '115', '118', '12', '120', '125', '127', '128', '13', '130', '131', '133', '14', '147', '148', '149', '15', '150', '152', '153', '155', '16', '161', '166', '167', '168', '169', '170', '172', '174', '175', '178', '18', '181', '183', '184', '185', '191', '194', '198', '2', '20', '21', '215', '216', '218', '22', '224', '225', '228', '229', '23', '

## 11. Interface utilisateur Streamlit

L'interface web est dans `streamlit_app.py`. Elle permet de saisir une question, regler le nombre de documents, afficher le prompt, comparer les 3 LLMs, voir precision/recall et consulter les references.

In [34]:
# Lancer dans un terminal depuis ce dossier:
# python -m streamlit run streamlit_app.py --server.headless=true --server.port=8501

print('Interface disponible avec la commande:')
print('python -m streamlit run streamlit_app.py --server.headless=true --server.port=8501')
print()
print('Pour le mode reel:')
print('- GPT: definir OPENAI_API_KEY, optionnellement OPENAI_MODEL')
print('- Qwen: installer/avoir transformers + torch, optionnellement QWEN_MODEL')
print('- Llama: lancer Ollama puis pull le modele, ex: ollama pull llama3.2')

Interface disponible avec la commande:
python -m streamlit run streamlit_app.py --server.headless=true --server.port=8501
